# 06 - Feature Engineering

## Objetivo

Este notebook construye el dataset definitivo de modelado a partir del dataset integrado generado en el Notebook 05.

La etapa de Feature Engineering transforma variables financieras existentes en variables explicativas adecuadas para modelos de Machine Learning, respetando estrictamente la regla temporal del proyecto.

### Regla temporal

Features ≤ t-1

Target = BTC_return(t+1)

Ninguna feature puede utilizar información contemporánea o futura respecto del objetivo de predicción.

### Output

dataset_features.csv

### Features Fase 1

- btc_return(t-1)
- btc_return(t-2)
- btc_volatility_7d(t-1)
- btc_ma_7d(t-1)
- dxy_return(t-1)
- gold_return(t-1)
- vix_Close(t-1)

### Target

- btc_return(t+1)

# 1. Librerías y configuración

Se importan las librerías necesarias y se define la ruta base del proyecto para cargar el dataset integrado generado en el Notebook 05.

In [1]:
import os
import pandas as pd
import numpy as np

PROJECT_PATH = r"C:\DS2_BTC_DXY_ORO_VIX"

dataset_path = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "dataset_integrado.csv"
)

# 2. Carga del dataset integrado

Se carga el dataset integrado generado en el Notebook 05.

Este dataset contiene las series de BTC, DXY, Oro y VIX alineadas temporalmente, y será utilizado como base para construir las features definitivas del modelo.

In [2]:
integrated_df = pd.read_csv(
    dataset_path,
    index_col=0,
    parse_dates=True
)

print("=== DATASET INTEGRADO CARGADO ===")
print(f"Shape: {integrated_df.shape}")
print(f"Rango temporal: {integrated_df.index.min().date()} → {integrated_df.index.max().date()}")

=== DATASET INTEGRADO CARGADO ===
Shape: (752, 32)
Rango temporal: 2023-05-08 → 2026-05-06


# 3. Validación inicial del dataset integrado

Antes de comenzar el proceso de Feature Engineering se realiza una validación básica del dataset integrado para verificar su estructura, rango temporal y presencia de valores faltantes.

Esta validación permite detectar posibles problemas antes de construir las variables predictivas y el target del modelo.

In [3]:
print("=== VALIDACIÓN INICIAL ===")

print(f"Shape: {integrated_df.shape}")
print()

print(f"Fecha inicial: {integrated_df.index.min().date()}")
print(f"Fecha final: {integrated_df.index.max().date()}")
print()

print("Valores nulos por columna:")
print(integrated_df.isna().sum())

=== VALIDACIÓN INICIAL ===
Shape: (752, 32)

Fecha inicial: 2023-05-08
Fecha final: 2026-05-06

Valores nulos por columna:
dxy_Close             0
dxy_High              0
dxy_Low               0
dxy_Open              0
dxy_Volume            0
dxy_return            1
dxy_volatility_7d     7
dxy_ma_7d             6
gold_Close            0
gold_High             0
gold_Low              0
gold_Open             0
gold_Volume           0
gold_return           1
gold_volatility_7d    7
gold_ma_7d            6
vix_Close             0
vix_High              0
vix_Low               0
vix_Open              0
vix_Volume            0
vix_return            1
vix_volatility_7d     7
vix_ma_7d             6
btc_Close             0
btc_High              0
btc_Low               0
btc_Open              0
btc_Volume            0
btc_return            0
btc_volatility_7d     5
btc_ma_7d             4
dtype: int64


# 4. Construcción del target

Se construye la variable objetivo del modelo.

El target corresponde al retorno de BTC del día siguiente (t+1), permitiendo que el modelo aprenda a predecir movimientos futuros utilizando exclusivamente información disponible hasta el día t.

In [4]:
features_df = integrated_df.copy()

features_df["target_btc_return_t1"] = (
    features_df["btc_return"].shift(-1)
)

In [5]:
features_df[
    ["btc_return", "target_btc_return_t1"]
].tail()

,btc_return,target_btc_return_t1
Date,,
2026-04-30,0.006970,0.024568
2026-05-01,0.024568,0.016421
2026-05-04,0.016421,0.013769
2026-05-05,0.013769,0.006184
2026-05-06,0.006184,NaN


# 5. Construcción de features predictivas

Se construyen las variables explicativas del modelo utilizando exclusivamente información disponible hasta el día anterior al objetivo de predicción.

Todas las features son desplazadas temporalmente para evitar filtración de información futura (data leakage).

In [6]:
features_df["btc_return_lag1"] = (
    features_df["btc_return"].shift(1)
)

features_df["btc_return_lag2"] = (
    features_df["btc_return"].shift(2)
)

features_df["btc_volatility_7d_lag1"] = (
    features_df["btc_volatility_7d"].shift(1)
)

features_df["btc_ma_7d_lag1"] = (
    features_df["btc_ma_7d"].shift(1)
)

features_df["dxy_return_lag1"] = (
    features_df["dxy_return"].shift(1)
)

features_df["gold_return_lag1"] = (
    features_df["gold_return"].shift(1)
)

features_df["vix_close_lag1"] = (
    features_df["vix_Close"].shift(1)
)

In [7]:
features_df[
    [
        "btc_return",
        "btc_return_lag1",
        "btc_return_lag2"
    ]
].head(10)

,btc_return,btc_return_lag1,btc_return_lag2
Date,,,
2023-05-08,-0.026734,NaN,NaN
2023-05-09,-0.001282,-0.026734,NaN
2023-05-10,-0.001338,-0.001282,-0.026734
2023-05-11,-0.022481,-0.001338,-0.001282
2023-05-12,-0.007252,-0.022481,-0.001338
2023-05-15,0.009731,-0.007252,-0.022481
2023-05-16,-0.005738,0.009731,-0.007252
2023-05-17,0.013395,-0.005738,0.009731
2023-05-18,-0.020680,0.013395,-0.005738


In [9]:
print("=== VALIDACIÓN DE NaN EXISTENTES ===")
print()

print(
    features_df[
        [
            "btc_return_lag1",
            "btc_return_lag2",
            "btc_volatility_7d_lag1",
            "btc_ma_7d_lag1",
            "dxy_return_lag1",
            "gold_return_lag1",
            "vix_close_lag1",
            "target_btc_return_t1"
        ]
    ].isna().sum()
)

=== VALIDACIÓN DE NaN EXISTENTES ===

btc_return_lag1           1
btc_return_lag2           2
btc_volatility_7d_lag1    6
btc_ma_7d_lag1            5
dxy_return_lag1           2
gold_return_lag1          2
vix_close_lag1            1
target_btc_return_t1      1
dtype: int64


# 6. Limpieza final del dataset

Luego de construir las features y el target se eliminan las filas que contienen valores faltantes.

Los NaN presentes son consecuencia esperada de los cálculos de retornos, ventanas móviles y desplazamientos temporales utilizados para evitar data leakage.

In [10]:
features_df = features_df.dropna().copy()

In [11]:
print("=== DATASET LIMPIO ===")
print()

print(f"Shape final: {features_df.shape}")
print()

print("NaN restantes:")
print(features_df.isna().sum().sum())

=== DATASET LIMPIO ===

Shape final: (744, 40)

NaN restantes:
0


# 7. Validación final

Se verifica la estructura final del dataset de modelado antes de exportarlo para las etapas posteriores del proyecto.

In [12]:
print("=== VALIDACIÓN FINAL ===")
print()

print(f"Shape final: {features_df.shape}")
print()

print("NaN restantes:")
print(features_df.isna().sum().sum())

print()
print("Columnas:")
print(features_df.columns.tolist())

=== VALIDACIÓN FINAL ===

Shape final: (744, 40)

NaN restantes:
0

Columnas:
['dxy_Close', 'dxy_High', 'dxy_Low', 'dxy_Open', 'dxy_Volume', 'dxy_return', 'dxy_volatility_7d', 'dxy_ma_7d', 'gold_Close', 'gold_High', 'gold_Low', 'gold_Open', 'gold_Volume', 'gold_return', 'gold_volatility_7d', 'gold_ma_7d', 'vix_Close', 'vix_High', 'vix_Low', 'vix_Open', 'vix_Volume', 'vix_return', 'vix_volatility_7d', 'vix_ma_7d', 'btc_Close', 'btc_High', 'btc_Low', 'btc_Open', 'btc_Volume', 'btc_return', 'btc_volatility_7d', 'btc_ma_7d', 'target_btc_return_t1', 'btc_return_lag1', 'btc_return_lag2', 'btc_volatility_7d_lag1', 'btc_ma_7d_lag1', 'dxy_return_lag1', 'gold_return_lag1', 'vix_close_lag1']


# 8. Exportación de datasets

Se exportan dos datasets:

- dataset_features_full.csv: contiene todas las variables originales, las features construidas y el target.
- dataset_modeling.csv: contiene únicamente las variables utilizadas por los modelos predictivos.

Esta separación permite conservar la trazabilidad completa del proceso de Feature Engineering y, al mismo tiempo, disponer de un dataset limpio y específico para las etapas de modelado.

## 8.1. Crear dataset de modelado

In [13]:
modeling_df = features_df[
    [
        "target_btc_return_t1",
        "btc_return_lag1",
        "btc_return_lag2",
        "btc_volatility_7d_lag1",
        "btc_ma_7d_lag1",
        "dxy_return_lag1",
        "gold_return_lag1",
        "vix_close_lag1"
    ]
].copy()

## 8.2. Validación

In [14]:
print("=== DATASET DE MODELADO ===")
print()

print(f"Shape: {modeling_df.shape}")
print()

print(modeling_df.head())

=== DATASET DE MODELADO ===

Shape: (744, 8)

            target_btc_return_t1  btc_return_lag1  btc_return_lag2  \
Date                                                                 
2023-05-17             -0.020680        -0.005738         0.009731   
2023-05-18              0.002159         0.013395        -0.005738   
2023-05-19              0.003643        -0.020680         0.013395   
2023-05-22              0.013945         0.002159        -0.020680   
2023-05-23             -0.032723         0.003643         0.002159   

            btc_volatility_7d_lag1  btc_ma_7d_lag1  dxy_return_lag1  \
Date                                                                  
2023-05-17                0.010379    27053.085100         0.001269   
2023-05-18                0.012160    27021.234654         0.003120   
2023-05-19                0.011640    26997.151786         0.006804   
2023-05-22                0.011314    27009.314453        -0.003669   
2023-05-23                0.012301   

## 8.3. Exportación de datasets

In [15]:
features_full_path = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "dataset_features_full.csv"
)

modeling_path = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "dataset_modeling.csv"
)

features_df.to_csv(features_full_path)
modeling_df.to_csv(modeling_path)

## 8.4. Validación de exportación

In [16]:
print("=== EXPORT COMPLETADO ===")
print()

print(f"dataset_features_full.csv -> {features_df.shape}")
print(f"dataset_modeling.csv      -> {modeling_df.shape}")

=== EXPORT COMPLETADO ===

dataset_features_full.csv -> (744, 40)
dataset_modeling.csv      -> (744, 8)


# 9. Conclusiones

En este notebook se construyó el dataset definitivo para las etapas de modelado a partir del dataset integrado generado previamente.

Se implementó la regla temporal del proyecto:

Features ≤ t-1

Target = BTC_return(t+1)

garantizando que todas las variables explicativas utilicen exclusivamente información disponible antes del objetivo de predicción y evitando cualquier forma de data leakage.

Como resultado se generaron dos datasets:

- dataset_features_full.csv: contiene las variables originales, las features construidas y el target.
- dataset_modeling.csv: contiene únicamente las variables utilizadas durante las etapas de entrenamiento y evaluación de modelos.

El proceso finalizó con un dataset limpio de 744 observaciones, sin valores faltantes y listo para ser utilizado en las etapas posteriores del proyecto.

## 9.1 Convención de nombres

Las columnas originales heredadas del dataset integrado conservan exactamente los nombres provenientes de las fuentes originales de datos.

Por este motivo pueden observarse variables como:

- btc_Close
- gold_Close
- dxy_Close
- vix_Close

Sin embargo, todas las variables construidas durante este proyecto siguen una convención homogénea basada en minúsculas y sufijos descriptivos, por ejemplo:

- btc_return_lag1
- btc_return_lag2
- btc_volatility_7d_lag1
- btc_ma_7d_lag1
- dxy_return_lag1
- gold_return_lag1
- vix_close_lag1
- target_btc_return_t1

Esta diferencia permite distinguir visualmente las variables originales de aquellas generadas durante el proceso de Feature Engineering.